# AstroShield: evolucionar una constelación inteligente

**Equipo:** CafeSmart
**Integrantes:** Moraleas Mendez Shyrley Nicole, Tejero Hernandez Rubi, Jimenez Andrade Monserrat, Sarao Morales Rubi · 
**Grupo:** LITIID-71 

Este notebook contiene los entregables de la actividad: la definición del problema (1), la implementación (2), las pruebas y la validación (3), el análisis de resultados (4) y la visualización 3D (5). El repositorio (6) incluye este notebook y su historial de commits.

| Entregable | Dónde está |
|---|---|
| 1. Definición del problema | Sección 1 |
| 2. Implementación funcional | Secciones 2 a 4 (código) |
| 3. Pruebas y validación | Secciones 7 a 15 |
| 4. Análisis de resultados | Sección 16 |
| 5. Visualización 3D | Se define en la sección 5 y se muestra en las secciones 7, 10, 11, 13 y 15 (globos interactivos: se pueden girar con el mouse) |
| 6. Repositorio, IA y limitaciones | Sección 17 y README |

In [ ]:
# Si falta una librería, instálala. 
# %pip install matplotlib pandas

## 1. Definición del problema (Entregable 1)

**Modelo simplificado.** La Tierra es una esfera de radio 6371 km. Un satélite es un punto de la esfera (su posición sobre la superficie), sin altura, órbita ni tiempo. Las distancias son de círculo máximo (fórmula haversine).

**1.1 Representación de una configuración.** Una matriz de 8 × 2: cada fila es un satélite con su latitud y longitud en grados. El orden de las filas no importa. Se eligió una representación de números reales y no una cuadrícula discreta porque la posición exacta de cada satélite define qué parte de una región queda cubierta.

**1.2 Cuándo una región está cubierta.**
- Cada satélite observa un disco de **250 km** de radio sobre la superficie (su *huella*). El enunciado solo fija el radio de 300 km de las regiones, así que la huella es una decisión del equipo: es menor que la región, para que ubicar con precisión y repartir los satélites importe.
- Cada región es un disco de 300 km alrededor de su centro, representado por 61 puntos repartidos de forma uniforme (espiral de girasol).
- La **cobertura** de una región es la fracción de esos puntos que está dentro de la huella de al menos un satélite.
- Una región está **cubierta** si su cobertura es al menos **0.5** (más de la mitad del área). Con un solo satélite centrado se cubre como máximo (250/300)² ≈ 69% de la región, así que añadir un segundo satélite sobre la misma región mejora su cobertura.

**1.3 Calidad de una configuración.** Puntuación = Σ (peso × cobertura) / Σ pesos, con un número entre 0 y 1. Los pesos son proporcionales a la prioridad: **Alta = 3, Media = 2, Normal = 1**. Esto significa que cubrir una región Alta vale el triple que una Normal, pero no que las demás se ignoren. Es una decisión del equipo: con otros pesos el reparto de satélites cambia (se comprueba al cambiar prioridades en la sección 12).

**1.4 Configuración inválida.** Es inválida si: (a) algún satélite queda dentro de una de las 3 zonas de desechos o de las 2 posiciones ocupadas (a menos distancia angular que su radio), o (b) no tiene la forma correcta (8 satélites, latitud en [-90, 90], longitud en [-180, 180], sin valores no numéricos). Una configuración inválida **nunca** se considera mejor que una válida, aunque su puntuación sea alta.

**1.5 Estrategia del algoritmo evolutivo.** Algoritmo genético con representación real:
- **Población inicial:** 80 configuraciones con posiciones aleatorias uniformes sobre la esfera (sin corregir las inválidas).
- **Selección:** torneo de 3 individuos con las reglas de Deb: válida contra inválida gana la válida; entre válidas, la de mayor puntuación; entre inválidas, la de menor violación (suma de cuánto penetra cada satélite en cada zona, como fracción del radio).
- **Cruce:** uniforme a nivel de satélite (cada satélite del hijo viene de uno de los dos padres), con probabilidad 0.9.
- **Mutación:** cada satélite muta con probabilidad 0.2, de cuatro formas: ajuste fino (60%, σ = 1°), salto medio (20%, σ = 10°), reubicación aleatoria en toda la esfera (10%) y salto cerca del centro de una región (10%, σ = 3°). Este último tipo se añadió porque, con huellas de 250 km, la probabilidad de que un satélite aleatorio caiga cerca de una región es de aproximadamente 1%; la sección 13 compara con y sin esta mutación.
- **Elitismo:** los 2 mejores pasan sin cambios. **Duración:** 150 generaciones (11 780 evaluaciones).

**1.6 Parámetros y valores iniciales.** Población 80, 150 generaciones, torneo 3, cruce 0.9, mutación 0.2 por satélite. Los valores iniciales se eligieron por razonamiento (población suficiente para cubrir la esfera, mutación que cambie en promedio 1 o 2 satélites por hijo) y después se comprobó su sensibilidad con ejecuciones reales (sección 13). Todos son configurables en `evolucionar(...)`.

**1.7 Cuándo es suficientemente buena una solución.** Se fijaron estos criterios sobre las 10 ejecuciones de referencia:
1. Todas las configuraciones obtenidas son válidas y el validador independiente no reporta errores.
2. Las regiones de prioridad Alta y Media quedan cubiertas en todas las ejecuciones.
3. La puntuación media supera claramente a una búsqueda aleatoria con el mismo número de evaluaciones.
4. La desviación estándar entre ejecuciones es menor o igual a 0.02.

Como referencia, se calcula una **cota de referencia** de la puntuación (sección 4): ningún algoritmo puede cubrir más puntos por satélite que los que cabe en una huella, así que se reparten los satélites entre regiones de la mejor forma posible ignorando traslapes. No se exige alcanzarla, solo se reporta la brecha.

**Supuestos y limitaciones del modelo:** sin órbitas ni tiempo (un satélite "ve" siempre su disco), huella fija y circular, tierra esférica, cobertura estimada con 61 puntos por región (resolución de 1/61 ≈ 1.6%), y pesos de prioridad definidos por el equipo.

## 2. Datos y geometría

Primero los datos del enunciado y las funciones geométricas (distancias de círculo máximo y puntos sobre la esfera).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import copy, itertools, math, time

R_TIERRA = 6371.0   # radio medio de la Tierra (km)

# ---------- Datos del enunciado ----------
ESCENARIO = {
    "n_sat": 8,
    "radio_obs_km": 300.0,        # radio de cada región (dato del enunciado)
    "huella_km": 250.0,           # radio que observa cada satélite (decisión del equipo)
    "umbral_cubierta": 0.5,       # fracción mínima para llamar "cubierta" a una región
    "regiones": [
        {"nombre": "Península de Yucatán", "lat": 20.5,  "lon": -89.0,  "prio": "Alta"},
        {"nombre": "Amazonía brasileña",   "lat": -3.5,  "lon": -62.0,  "prio": "Alta"},
        {"nombre": "California",           "lat": 36.8,  "lon": -119.4, "prio": "Media"},
        {"nombre": "Sahel",                "lat": 15.0,  "lon": 0.0,    "prio": "Media"},
        {"nombre": "Japón",                "lat": 36.0,  "lon": 138.0,  "prio": "Media"},
        {"nombre": "Interior de Australia","lat": -25.0, "lon": 133.0,  "prio": "Normal"},
    ],
    "zonas_desechos": [
        {"nombre": "Atlántico Norte", "lat": 45.0,  "lon": -35.0,  "radio_deg": 12.0},
        {"nombre": "Pacífico Sur",    "lat": -30.0, "lon": -120.0, "radio_deg": 10.0},
        {"nombre": "Región polar norte", "lat": 78.0, "lon": 30.0, "radio_deg": 8.0},
    ],
    "ocupadas": [
        {"nombre": "Objeto A", "lat": 10.0,  "lon": 45.0,  "radio_deg": 5.0},
        {"nombre": "Objeto B", "lat": -15.0, "lon": 110.0, "radio_deg": 5.0},
    ],
}
PESO = {"Alta": 3, "Media": 2, "Normal": 1}   # prioridad -> peso en la puntuación

In [ ]:
def a_vec(lat, lon):
    """(lat, lon) en grados -> vector unitario 3D."""
    la, lo = np.radians(lat), np.radians(lon)
    return np.stack([np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)], axis=-1)

def dist_km(lat1, lon1, lat2, lon2):
    """Distancia de círculo máximo (haversine) en km. Acepta arreglos."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dlam = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2) ** 2
    return 2 * R_TIERRA * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

def destino(lat, lon, d_km, rumbo):
    """Punto a d_km del origen siguiendo el rumbo (rad). Devuelve (lat, lon) en grados."""
    d, p1, l1 = np.asarray(d_km) / R_TIERRA, np.radians(lat), np.radians(lon)
    p2 = np.arcsin(np.sin(p1) * np.cos(d) + np.cos(p1) * np.sin(d) * np.cos(rumbo))
    l2 = l1 + np.arctan2(np.sin(rumbo) * np.sin(d) * np.cos(p1), np.cos(d) - np.sin(p1) * np.sin(p2))
    return np.degrees(p2), (np.degrees(l2) + 180.0) % 360.0 - 180.0

def puntos_region(reg, radio_km, n=61):
    """n puntos repartidos de forma uniforme (espiral de girasol) dentro del disco de la región."""
    k = np.arange(n)
    r = radio_km * np.sqrt((k + 0.5) / n)
    theta = k * np.pi * (3 - np.sqrt(5))
    return destino(reg["lat"], reg["lon"], r, theta)

def aleatorias(rng, n, S):
    """n configuraciones de S satélites con posiciones uniformes sobre la esfera. Forma (n, S, 2)."""
    lat = np.degrees(np.arcsin(rng.uniform(-1, 1, (n, S))))
    lon = rng.uniform(-180, 180, (n, S))
    return np.stack([lat, lon], axis=-1)

## 3. Evaluación, restricciones y validación

La clase `Problema` calcula la cobertura, la puntuación y la violación de restricciones de muchas configuraciones a la vez. `validar_config` es un **validador independiente**: usa vectores 3D y `arccos`, sin reutilizar el evaluador ni la fórmula haversine, para que un error en uno no pase desapercibido en el otro.

In [ ]:
class Problema:
    """Convierte un escenario en un problema evaluable: cobertura, puntuación y violación de restricciones."""
    def __init__(self, esc):
        self.esc = esc
        self.n_sat = esc["n_sat"]
        self.huella = esc["huella_km"]
        self.radio_obs = esc["radio_obs_km"]
        self.umbral = esc["umbral_cubierta"]
        regs = esc["regiones"]
        self.nombres = [r["nombre"] for r in regs]
        self.prios = [r["prio"] for r in regs]
        self.w = np.array([PESO[r["prio"]] for r in regs], float)
        pts = [puntos_region(r, self.radio_obs) for r in regs]
        self.p_lat = np.array([p[0] for p in pts])      # (R, n_puntos)
        self.p_lon = np.array([p[1] for p in pts])
        self.c_lat = np.array([r["lat"] for r in regs])
        self.c_lon = np.array([r["lon"] for r in regs])
        zs = list(esc["zonas_desechos"]) + list(esc["ocupadas"])
        self.z_nombre = [z["nombre"] for z in zs]
        self.z_lat = np.array([z["lat"] for z in zs], float)
        self.z_lon = np.array([z["lon"] for z in zs], float)
        self.z_rad = np.array([z["radio_deg"] for z in zs], float)
        self.dist = dist_km                              # se puede reemplazar en pruebas (mutantes)

    def cobertura(self, P):
        """P: (N, S, 2) -> fracción cubierta de cada región, (N, R)."""
        lat, lon = P[:, :, 0][:, :, None, None], P[:, :, 1][:, :, None, None]
        d = self.dist(lat, lon, self.p_lat[None, None], self.p_lon[None, None])   # (N, S, R, n)
        return (d <= self.huella).any(axis=1).mean(axis=2)

    def puntuacion(self, cob):
        return (cob * self.w).sum(axis=-1) / self.w.sum()

    def violacion(self, P):
        """Suma de cuánto penetra cada satélite en cada zona prohibida (como fracción del radio). 0 = válida."""
        if len(self.z_rad) == 0:
            return np.zeros(len(P))
        d_km = self.dist(P[:, :, 0][:, :, None], P[:, :, 1][:, :, None], self.z_lat[None, None], self.z_lon[None, None])
        d_deg = np.degrees(d_km / R_TIERRA)
        return (np.maximum(0.0, self.z_rad - d_deg) / self.z_rad).sum(axis=(1, 2))

    def evaluar(self, P):
        P = np.asarray(P, float)
        if P.ndim == 2:
            P = P[None]
        return self.puntuacion(self.cobertura(P)), self.violacion(P)

def validar_config(esc, cfg):
    """Validador independiente (vectores 3D y arccos, sin usar haversine ni el evaluador). Devuelve lista de errores."""
    cfg = np.asarray(cfg, float)
    if cfg.shape != (esc["n_sat"], 2):
        return [f"forma {cfg.shape}, se esperaba ({esc['n_sat']}, 2)"]
    if not np.isfinite(cfg).all():
        return ["hay valores no numéricos"]
    errs = []
    if (np.abs(cfg[:, 0]) > 90).any():
        errs.append("latitud fuera de [-90, 90]")
    if (np.abs(cfg[:, 1]) > 180).any():
        errs.append("longitud fuera de [-180, 180]")
    v = a_vec(cfg[:, 0], cfg[:, 1])
    for z in list(esc["zonas_desechos"]) + list(esc["ocupadas"]):
        ang = np.degrees(np.arccos(np.clip(v @ a_vec(z["lat"], z["lon"]), -1, 1)))
        for i, a in enumerate(ang):
            if a < z["radio_deg"] - 1e-9:
                errs.append(f"S{i+1} dentro de «{z['nombre']}» ({a:.2f}° < {z['radio_deg']}°)")
    return errs

def margenes(esc, cfg):
    """Distancia mínima de la constelación a cada zona restringida, con el mismo método del validador."""
    v = a_vec(np.asarray(cfg)[:, 0], np.asarray(cfg)[:, 1])
    filas = []
    for tipo, lista in (("Desechos", esc["zonas_desechos"]), ("Ocupada", esc["ocupadas"])):
        for z in lista:
            ang = np.degrees(np.arccos(np.clip(v @ a_vec(z["lat"], z["lon"]), -1, 1)))
            filas.append({"Tipo": tipo, "Zona": z["nombre"], "Radio (°)": z["radio_deg"],
                          "Satélite más cercano (°)": round(float(ang.min()), 2),
                          "Margen (°)": round(float(ang.min() - z["radio_deg"]), 2),
                          "Cumple": bool(ang.min() >= z["radio_deg"] - 1e-9)})
    return pd.DataFrame(filas)

## 4. Algoritmo evolutivo, línea base y cota de referencia

`evolucionar` implementa el algoritmo de la sección 1.5. Se incluyen también una búsqueda aleatoria con el mismo número de evaluaciones (línea base para comparar), `informe` (tabla de cobertura por región), `cota_referencia` y `resolver` (que informa explícitamente cuando el escenario es inviable).

In [2]:
def clave(score, viol):
    """Regla de Deb: toda configuración válida es mejor que cualquier inválida.
    Válidas se ordenan por puntuación (clave = -puntuación, en [-1, 0]); inválidas por violación (clave = 1 + violación)."""
    return np.where(viol <= 0, -score, 1.0 + viol)

def mutar(rng, H, prob, p_mut, sig_fino, sig_medio, mezcla):
    """Mutación por satélite (con probabilidad p_mut). Cuatro tipos, elegidos según `mezcla`:
    0 ajuste fino (gaussiana pequeña), 1 salto medio (gaussiana grande),
    2 reubicación aleatoria en toda la esfera, 3 salto cerca del centro de una región."""
    n, S, _ = H.shape
    lat, lon = H[..., 0], H[..., 1]
    muta = rng.random((n, S)) < p_mut
    tipo = rng.choice(4, size=(n, S), p=mezcla)
    sig = np.where(tipo == 0, sig_fino, sig_medio)
    nlat = np.clip(lat + rng.normal(0, 1, (n, S)) * sig, -90, 90)
    nlon = (lon + rng.normal(0, 1, (n, S)) * sig / np.maximum(np.cos(np.radians(lat)), 0.1) + 180) % 360 - 180
    rnd = aleatorias(rng, n, S)
    nlat, nlon = np.where(tipo == 2, rnd[..., 0], nlat), np.where(tipo == 2, rnd[..., 1], nlon)
    ir = rng.integers(0, len(prob.c_lat), (n, S))
    jlat = np.clip(prob.c_lat[ir] + rng.normal(0, 3, (n, S)), -90, 90)
    jlon = (prob.c_lon[ir] + rng.normal(0, 3, (n, S)) / np.maximum(np.cos(np.radians(jlat)), 0.1) + 180) % 360 - 180
    nlat, nlon = np.where(tipo == 3, jlat, nlat), np.where(tipo == 3, jlon, nlon)
    return np.stack([np.where(muta, nlat, lat), np.where(muta, nlon, lon)], axis=-1)

def evolucionar(prob, pop=80, gens=150, elite=2, k_torneo=3, p_cruce=0.9, p_mut=0.2,
                sig_fino=1.0, sig_medio=10.0, mezcla=(0.6, 0.2, 0.1, 0.1), seed=0):
    rng = np.random.default_rng(seed)
    S = prob.n_sat
    P = aleatorias(rng, pop, S)
    sc, vi = prob.evaluar(P)
    evals = pop
    hist, cfg_ini = [], None
    for g in range(gens + 1):
        key = clave(sc, vi)
        ib = int(np.argmin(key))
        valida = vi <= 0
        hist.append({"gen": g, "clave_mejor": key[ib], "mejor": sc[ib], "viol_mejor": vi[ib],
                     "mejor_valida": sc[valida].max() if valida.any() else np.nan,
                     "media": sc.mean(), "pct_validas": 100 * valida.mean()})
        if g == 0:
            cfg_ini, sc_ini, vi_ini = P[ib].copy(), sc[ib], vi[ib]
        if g == gens:
            break
        orden = np.argsort(key)
        n_h = pop - elite
        cand = rng.integers(0, pop, size=(2 * n_h, k_torneo))                    # torneo
        gan = cand[np.arange(2 * n_h), np.argmin(key[cand], axis=1)]
        pa, pb = P[gan[:n_h]], P[gan[n_h:]]
        mascara = rng.random((n_h, S, 1)) < 0.5                                   # cruce uniforme por satélite
        cruzar = rng.random((n_h, 1, 1)) < p_cruce
        hijos = mutar(rng, np.where(cruzar & mascara, pb, pa), prob, p_mut, sig_fino, sig_medio, mezcla)
        hs, hv = prob.evaluar(hijos)
        evals += n_h
        P = np.concatenate([P[orden[:elite]], hijos])                             # elitismo
        sc = np.concatenate([sc[orden[:elite]], hs])
        vi = np.concatenate([vi[orden[:elite]], hv])
    key = clave(sc, vi)
    ib = int(np.argmin(key))
    return {"cfg": P[ib].copy(), "score": float(sc[ib]), "viol": float(vi[ib]), "valida": bool(vi[ib] <= 0),
            "cfg_ini": cfg_ini, "score_ini": float(sc_ini), "viol_ini": float(vi_ini),
            "hist": pd.DataFrame(hist), "evals": evals, "seed": seed}

def busqueda_aleatoria(prob, n_eval, seed=0, lote=2000):
    """Línea base: muestrear configuraciones al azar con el mismo número de evaluaciones y quedarse con la mejor."""
    rng = np.random.default_rng(seed)
    mejor = (np.inf, None, 0.0, 0.0)
    hecho = 0
    while hecho < n_eval:
        m = min(lote, n_eval - hecho)
        P = aleatorias(rng, m, prob.n_sat)
        sc, vi = prob.evaluar(P)
        k = clave(sc, vi)
        i = int(np.argmin(k))
        if k[i] < mejor[0]:
            mejor = (k[i], P[i].copy(), float(sc[i]), float(vi[i]))
        hecho += m
    return {"cfg": mejor[1], "score": mejor[2], "viol": mejor[3], "valida": mejor[3] <= 0, "evals": n_eval, "seed": seed}

def informe(prob, cfg):
    """Tabla de cobertura por región para una configuración."""
    cfg = np.asarray(cfg, float)
    cob = prob.cobertura(cfg[None])[0]
    filas = []
    for i, nombre in enumerate(prob.nombres):
        d = prob.dist(cfg[:, 0, None], cfg[:, 1, None], prob.p_lat[i][None], prob.p_lon[i][None])
        filas.append({"Región": nombre, "Prioridad": prob.prios[i], "Cobertura (%)": round(100 * cob[i], 1),
                      "Estado": "Cubierta" if cob[i] >= prob.umbral else "NO cubierta",
                      "Satélites que la observan": int((d <= prob.huella).any(axis=1).sum())})
    return pd.DataFrame(filas)

def cota_referencia(prob):
    """Cota superior de referencia: ningún satélite cubre más puntos de una región que c_max (calculado sobre
    una malla de posiciones); con k satélites en la región se cubre a lo sumo k*c_max. Se reparten los
    satélites entre regiones de la mejor forma posible, ignorando traslapes entre huellas."""
    g = np.arange(-prob.huella - 100, prob.huella + 101, 10.0)
    gx, gy = np.meshgrid(g, g)
    dd, br = np.hypot(gx, gy).ravel(), np.arctan2(gx, gy).ravel()
    a = []
    for i in range(len(prob.w)):
        la, lo = destino(prob.c_lat[i], prob.c_lon[i], dd, br)
        d = prob.dist(la[:, None], lo[:, None], prob.p_lat[i][None], prob.p_lon[i][None])
        a.append((d <= prob.huella).sum(axis=1).max() / prob.p_lat.shape[1])
    R, mejor = len(prob.w), [0.0]
    def rec(i, resto, acc):
        if i == R - 1:
            mejor[0] = max(mejor[0], acc + prob.w[i] * min(1.0, resto * a[i]))
            return
        for k in range(resto + 1):
            rec(i + 1, resto - k, acc + prob.w[i] * min(1.0, k * a[i]))
    rec(0, prob.n_sat, 0.0)
    return mejor[0] / prob.w.sum(), a

def asignacion(prob, cfg):
    """Cuántos satélites observan cada región (huella que toca al menos un punto de la región)."""
    return informe(prob, cfg)["Satélites que la observan"].tolist()

def resolver(esc, **kw):
    """Resuelve un escenario completo. Si no existe configuración válida, lo dice explícitamente."""
    pr = Problema(esc)
    r = evolucionar(pr, **kw)
    r["problema"] = pr
    r["mensaje"] = ("Configuración válida." if r["valida"] else
                    "INVIABLE: ninguna configuración cumple las restricciones; se muestra la de menor violación (NO es utilizable).")
    return r